# 0.  Introduction to PlatoSim

Welcome to PlatoSim! In this introductory tutorial we show how you can get started generating simulations. Furthermore we give an overview of all the utilties and help functions that are build into PlatoSim to help you getting started - enjoy!


<img src="../../figures/LogoPlatoSim.png" width="750"/>

### Setup notebook

In [1]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

### Imports

In [ ]:
import os

# PlatoSim
from platosim.utilities    import getFunctions
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

---
## 0.1 - Setup and run a simulation
---

As part of the installation of PlatoSim the project directory and working directory should already have been set. We don't need these paths for this series of tutorials, however, they are very handy when you start to create your own scripts. Thus check that these are globally defined (and if not, please first export them before continuing):

In [ ]:
projectDir = os.environ['PLATO_PROJECT_HOME']
workDir    = os.environ['PLATO_WORKDIR']
print(projectDir)
print(workDir)

First we define the input/output (I/O) paths and files needed for PlatoSim.

In [ ]:
# We use the default YAML input file and configure is later
inputDir  = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
inputFile = inputDir + "/inputfile.yaml"

# Save all output to current working directory
outputDir = os.getcwd()

Now a simulation object can be created. Note that by default PlatoSim used the YAML input file located at `PLATO_PROJECT_HOME`, however, we here show how to include it directly for the construction of the simulation object: 

In [ ]:
# Set up a Simulation object
outputFileName = "output_example1"
sim = Simulation(outputFileName, inputFile, outputDir=outputDir)

Note that you can set the output directory afterwards as well using 

```
sim.outputDir = outputDir
```

Also if not specified, the `inputfile.yaml` configuration file from the `/inputfiles` directory will be used (provided you have exported the `PLATO_PROJECT_HOME` environment variable). It is possible to also set the path to the reference input file as follows: 

```
sim.readConfigurationFile(<full/path/to/configuration/file>)
```

However, for both parameters, choose only one method since setting both will crash PlaotSim upon execution.

For now we use the default YAML settings to run our simulation. Note that by default PlatoSim do not overwrite the output from a previous simulation defined by the same name. To avoid that PlatoSim ends in errors, we here activate the `removeOutputFile` option when launching PlatoSim. Furthermore you can tract the execution time activating the parameter `executionTime` as done here: 

In [ ]:
# Run PlatoSim
simFile = sim.run(removeOutputFile=True, executionTime=True)

**Congratulations, you have a now generated your first PlatoSim simulation with Python!**

In this directory, the following information will be stored when running the simulation:
- `runName.yaml`: copy of the input file with the chosen configuration parameters (copied from the reference configuration file and possibly modified, as shown below);
- `runName.hdf5`: resulting exposures (images, PSF, etc.);
- `runName.log`: log file (to report any problems).


---
## 0.2 - Configuring the YAML input file
---

We note that all the configuration parameters in the YAML input file is the topic of the following tutorial: **01_ConfigurationParametersYAML**. Now say that we want to configure the input parameters but we don't want to open up the YAML input file to do so (e.g. because we want to make an automated script for our simulations). Again first we create an simulation object:

In [ ]:
# Initialise PlatoSim
outputFileName = "output_example2"
sim = Simulation(outputFileName, outputDir=outputDir)

Since we are persistent and really don't want to open the YAML file manually, let's have a look at it's content from within Python. You can do this using the `simfile.getYamlConfiguration()` function directly: 

In [ ]:
# Open and print content
sim.showYamlConfiguration()

Note that the output (of `pyaml`) shown here is alphabetically ordered, however, the real YAML input file has a different structure (so don't panic). After an overview of the input parameters let's assume we want to change the following parameters:

In [ ]:
# Observation
sim["ObservingParameters/NumExposures"] = 10

# Sky
sim["Sky/SkyBackground"]         = -1
sim["Sky/Cosmics/CosmicHitRate"] = 10

# Subfield
sim["SubField/NumColumns"]      = 300
sim["SubField/NumRows"]         = 300
sim["SubField/ZeroPointColumn"] = 0
sim["SubField/ZeroPointRow"]    = 0

We have chosen to simulate only 10 image, set the sky background to be estimated from an automatic tabulation according to the sky location (done with the `-1` flag), and we reduced the cosmic ray hit rate to "quiet" space weather conditions for PLATO (according to the results of the PLATO working groups). The subfield is enlarged but we keep the simulation in the origin of the CCD.

To gain the full control over what is being saved to the HDF5 output file (without the need to open the input YAML file), you can manually set all entries in the `ControlHDF5Content` to `no` (or `False`). However, for must use cases you are only interested in saving a few of these since the more that is saved to the HDF5, the higher the execution time for PlatoSim. Thus, a more efficient way is to specifically *turn off* the writing of all outputs using the function `turnOffAllOutput()` (and opposite you can *turn on* the writing of all output using `turnOnAllOutput()`) and then activate the writing of the entries you are intered in. Say we only want to save the pixel maps and star positions:

In [ ]:
# Turn off saving 
sim.turnOffAllOutput()

# Control HDF5
sim["ControlHDF5Content/WritePixelMaps"]     = True
sim["ControlHDF5Content/WriteStarPositions"] = True

Let's run the simulation:

In [ ]:
# Run PlatoSim
simfile = sim.run(removeOutputFile=True)

---
## 0.3 - Fetching the HDF5 output file
---

To access the HDF5 output file that contains your simulated data, we can simply use the `SimFile` class:

In [ ]:
# Fetch HDF5 output
f = SimFile("output_example2" + ".hdf5")

---
## 0.4 - Plot the simulated subfield
---

Plotting your simulated pixel images can be done out-of-box using the `showImage` utility of the `SimFile` class. This function takes can plot either a specific frame using `imageNr` or the entire image cube with an interactive slider below the plot as follows:

In [ ]:
fig, ax = f.showImage(imageNr=False, clipPercentile=1, imgScale="clip",  
                      figsize=(7,7), fontSize=15, useTitle=False,
                      showStarPositions=False, showStarIDs=False,
                      colorMap="cubehelix", colorBar=True, showGrid=True) 

---
## 0.5 - Overview of PlatoSim's class and utility functions
---

We have now covered the basics of using PlatoSim within a Python interface. The following is an overview of all the help functions and utilities that are directly build into the PlatoSim software package to ease your life of running, analyzing, and exploring your simulations. Below each Python class or script is a file located in the `python/platosim/` directory. Notice that an exploration of the following Python classes is integrated into the topics of the following tutorials.

In [ ]:
getFunctions(Simulation)

The functions of the `SimFile` class can be used to **retrieve, plot, and save information** about your simulation:

In [ ]:
getFunctions(SimFile)

The functions of the `referenceFrames` utility are **coordinate transformations** used internally by PlatoSim (hence generally they are often not needed to setup your simulations):

In [ ]:
import platosim.referenceFrames as rf 
getFunctions(rf)